# PlantDoc AI — Improved Training Workflow (Kaggle)

**Mục tiêu:** Huấn luyện mô hình PlantDocAI trên **Kaggle GPU** bằng workflow chuẩn hóa mới nhất của project.
Notebook này đóng vai trò **điều phối** (script-first), không chứa full training loop.

**Luồng thực thi:**
1. Chuẩn bị môi trường & code từ GitHub (hoặc attach Kaggle dataset).
2. Thiết lập đường dẫn Kaggle (`/kaggle/input/` và `/kaggle/working/`).
3. Tự động kiểm tra project & script.
4. Gọi script `scripts/createSplits.py` để tạo split mới (train/val/test) vào thư mục working.
5. Tự động patch file config (ví dụ: `configs/efficientnet.yaml`) trỏ tới dữ liệu Kaggle.
6. Gọi script `scripts/train.py` để huấn luyện mô hình.
7. Hỗ trợ **Resume Training**: tự động nhận diện `last.pt` từ attach output lần trước (nếu có).

> **Lưu ý quan trọng trên Kaggle:** File lưu trong `/kaggle/working/` sẽ bị xóa nếu bạn không **Save Version** (chọn Save & Run All). Hãy đảm bảo lưu phiên bản sau khi train xong!

## 1. Setup Environment
Kiểm tra GPU và chuẩn bị code dự án.

In [ ]:
!nvidia-smi

import os
import sys
import subprocess
from pathlib import Path

# Clone project từ GitHub
GIT_URL = "https://github.com/bien3008/PlantDocAI.git"
GIT_BRANCH = "experiment/fine-tune-model"
PROJECT_DIR = "/kaggle/working/PlantDocAI"

if not os.path.exists(PROJECT_DIR):
    print(f"📦 Cloning repository từ branch {GIT_BRANCH}...")
    subprocess.run(f"git clone --branch {GIT_BRANCH} {GIT_URL} {PROJECT_DIR}", shell=True, check=True)
else:
    print("✅ Repository đã tồn tại.")

os.chdir(PROJECT_DIR)
sys.path.append(PROJECT_DIR)
print("Current working directory:", os.getcwd())

# Cài đặt dependencies (bỏ qua các thư viện đã có sẵn trên Kaggle để tiết kiệm thời gian)
!pip install -q timm pyyaml
print("✅ Đã setup xong môi trường.")

## 2. Cấu hình biến môi trường và Đường dẫn Kaggle
Kaggle Dataset thường nằm trong `/kaggle/input/`. Thư mục làm việc là `/kaggle/working/`.

In [ ]:
# ==========================================
# CẤU HÌNH ĐƯỜNG DẪN & THAM SỐ (TÙY CHỈNH Ở ĐÂY)
# ==========================================

# 1. Dataset & Checkpoints Input
# ⚠️ THAY ĐỔI ĐƯỜNG DẪN NÀY CHO ĐÚNG VỚI TÊN DATASET BẠN ĐÃ ADD VÀO KAGGLE
KAGGLE_DATASET_ROOT = "/kaggle/input/plantdocai-extended" 

# Nếu bạn muốn resume từ lần chạy trước, hãy đính kèm Output của phiên bản trước vào Kaggle
# và sửa đường dẫn này trỏ tới thư mục chứa last.pt
PREVIOUS_CHECKPOINT_DIR = "/kaggle/input/plantdocai-previous-output/artifacts/efficientnetB0_extended_kaggle/checkpoints"

# 2. Training Config
BASE_CONFIG = "configs/efficientnet.yaml" # Dùng config có sẵn của project
EXPERIMENT_NAME = "efficientnetB0_extended_kaggle"
RESUME_TRAINING = True

# ==========================================
# KHÔNG CẦN CHỈNH SỬA BÊN DƯỚI
# ==========================================
KAGGLE_WORKING_DIR = Path("/kaggle/working")
SPLIT_DIR = KAGGLE_WORKING_DIR / "splits"
OUTPUT_DIR = KAGGLE_WORKING_DIR / "artifacts" / EXPERIMENT_NAME

SPLIT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
Path(OUTPUT_DIR / "checkpoints").mkdir(parents=True, exist_ok=True)

# Hàm tự động tìm root data chứa class folders
def find_data_root(base_dir: Path) -> str:
    # Tìm TẤT CẢ các thư mục chứa '___'
    class_dirs = [d for d in base_dir.rglob('*') if d.is_dir() and '___' in d.name]
    if not class_dirs:
        raise FileNotFoundError(f"Không tìm thấy class folders (chứa '___') trong: {base_dir}")
    
    # Gom nhóm theo thư mục cha để xem thư mục nào chứa nhiều class nhất
    parent_counts = {}
    for d in class_dirs:
        parent = str(d.parent)
        parent_counts[parent] = parent_counts.get(parent, 0) + 1
        
    # Chọn thư mục cha có nhiều class folders nhất (ví dụ: chứa đủ 38 classes)
    best_parent = max(parent_counts.items(), key=lambda x: x[1])[0]
    print(f"ℹ️ Tìm thấy các data roots tiềm năng: {parent_counts}")
    print(f"ℹ️ Chọn root có nhiều class nhất: {best_parent} ({parent_counts[best_parent]} classes)")
    return best_parent

try:
    DATA_DIR = find_data_root(Path(KAGGLE_DATASET_ROOT))
    print(f"✅ Data Root tìm thấy: {DATA_DIR}")
except Exception as e:
    print(f"❌ LỖI: {e}")
    print("Gợi ý: Hãy đảm bảo bạn đã 'Add Data' trên Kaggle và sửa biến KAGGLE_DATASET_ROOT cho đúng.")
    raise e

## 3. Verify Project Files
Đảm bảo các scripts quan trọng tồn tại.

In [ ]:
print("=== KIỂM TRA SCRIPTS VÀ CONFIG ===")
assert os.path.exists("scripts/train.py"), "❌ Không tìm thấy scripts/train.py!"
assert os.path.exists("scripts/createSplits.py"), "❌ Không tìm thấy scripts/createSplits.py!"
assert os.path.exists(BASE_CONFIG), f"❌ Không tìm thấy config {BASE_CONFIG}!"
print("✅ Project files đầy đủ.")

## 4. Tạo Data Splits Mới
Gọi script của project để tạo split (train, val, test, classes) lưu vào `/kaggle/working/splits`.

In [ ]:
print("=== TẠO SPLIT MỚI TỪ DATASET ===")
import shutil

# Xóa splits cũ nếu có
if SPLIT_DIR.exists():
    for f in ["train.csv", "val.csv", "test.csv", "classes.csv"]:
        target = SPLIT_DIR / f
        if target.exists():
            target.unlink()

!python scripts/createSplits.py --dataDir "{DATA_DIR}" --outDir "{SPLIT_DIR}"

print("\n=== KIỂM TRA SPLIT SAU KHI TẠO ===")
for f in ["train.csv", "val.csv", "test.csv", "classes.csv"]:
    p = SPLIT_DIR / f
    if p.exists():
        with open(p, 'r') as fp:
            line_count = sum(1 for _ in fp) - 1
        print(f"  ✅ {f}: {line_count} samples/classes")
    else:
        print(f"  ❌ {f}: THIẾU")

## 5. Cấu hình Config cho Kaggle
Tự động patch config YAML (EfficientNet B0) để nhận Kaggle Paths.

In [ ]:
import yaml

with open(BASE_CONFIG, "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

# Cập nhật paths sang Kaggle
config["dataDir"] = str(DATA_DIR)
config["splitDir"] = str(SPLIT_DIR)
config["outputDir"] = str(OUTPUT_DIR)
config["experimentName"] = EXPERIMENT_NAME

kaggle_config_path = Path("configs/kaggle_train_config.yaml")
with open(kaggle_config_path, "w", encoding="utf-8") as f:
    yaml.safe_dump(config, f, allow_unicode=True, sort_keys=False)

print(f"✅ Đã tạo config patch cho Kaggle tại: {kaggle_config_path}")
print("Config Paths Updated:")
print(f"  - dataDir: {config['dataDir']}")
print(f"  - splitDir: {config['splitDir']}")
print(f"  - outputDir: {config['outputDir']}")
print(f"  - modelName: {config['modelName']}")

## 6. Huấn luyện Mô hình
Sử dụng `scripts/train.py`. Nếu muốn resume từ lần chạy Kaggle trước, notebook sẽ tự copy `last.pt` từ input dataset vào working directory.

In [ ]:
print("=== BẮT ĐẦU HUẤN LUYỆN DỰA TRÊN SCRIPT ===")

resume_flag = "--resume" if RESUME_TRAINING else ""
working_ckpt_dir = OUTPUT_DIR / "checkpoints"
last_ckpt_working = working_ckpt_dir / "last.pt"

# Xử lý Resume trên Kaggle: Nếu có thư mục checkpoint từ Kaggle Dataset (lần trước), copy qua working
if RESUME_TRAINING:
    prev_ckpt_path = Path(PREVIOUS_CHECKPOINT_DIR) / "last.pt"
    if not last_ckpt_working.exists() and prev_ckpt_path.exists():
        print(f"🔍 Đã tìm thấy checkpoint cũ tại Kaggle Input: {prev_ckpt_path}")
        print(f"📦 Đang copy checkpoint sang thư mục working: {last_ckpt_working}...")
        shutil.copy2(prev_ckpt_path, last_ckpt_working)
        print("✅ Copy hoàn tất.")

if RESUME_TRAINING and last_ckpt_working.exists():
    print(f"🔍 Sẵn sàng resume từ: {last_ckpt_working}")
else:
    print("ℹ️ Bắt đầu huấn luyện mới hoàn toàn.")

# Thực thi script train (Không dùng --useGdrive vì Kaggle không dùng Drive)
!python scripts/train.py --config configs/kaggle_train_config.yaml {resume_flag}

## 7. Sanity Check Kết quả
Kiểm tra xem các model và logs đã lưu vào working directory thành công chưa.

In [ ]:
print("=== KIỂM TRA ARTIFACTS ĐẦU RA ===")

checkpoints_dir = OUTPUT_DIR / "checkpoints"
logs_dir = OUTPUT_DIR / "logs"
config_file = OUTPUT_DIR / "config.json"

print(f"Artifacts lưu tại: {OUTPUT_DIR}")

if config_file.exists():
    print("  ✅ config.json có tồn tại")
else:
    print("  ❌ Thiếu config.json")

if checkpoints_dir.exists():
    ckpts = list(checkpoints_dir.glob("*.pt"))
    print(f"  ✅ Tìm thấy {len(ckpts)} checkpoint(s): {[c.name for c in ckpts]}")
else:
    print("  ❌ Không tìm thấy thư mục checkpoints")
    
if logs_dir.exists():
    logs = list(logs_dir.glob("*.csv"))
    print(f"  ✅ Tìm thấy {len(logs)} log(s): {[l.name for l in logs]}")
else:
    print("  ❌ Không tìm thấy thư mục logs")

print("\n🎉 WORKFLOW HOÀN TẤT!")
print("⚠️ ĐỪNG QUÊN NHẤN 'Save Version' ĐỂ GIỮ LẠI OUTPUT TRONG KAGGLE KHI TẮT BROWSER!")